# MiniMind 配置一览(精简版)

> 本文是 [ch03.ipynb](./ch03.ipynb) 的浓缩版。只保留配置一览表和参数量公式,方便快速查阅。

## Config 一览表

| 超参数 | 值 | 说明 |
|---|---|---|
| `hidden_size` | 768 | 隐藏维度 $d$ |
| `num_hidden_layers` | 8 | Transformer 层数 $L$ |
| `vocab_size` | 6400 | 词表大小 $V$ |
| `num_attention_heads` | 8 | Q 头数 $n_q$ |
| `num_key_value_heads` | 4 | KV 头数 $n_{kv}$(GQA) |
| `head_dim` | 96 | 每头维度 $d_h = d / n_q$ |
| `intermediate_size` | 2432 | FFN 宽度 $= \lceil d\pi/64 \rceil \times 64$ |
| `hidden_act` | silu | SwiGLU 激活 |
| `max_position_embeddings` | 32768 | 最大序列长度 |
| `rope_theta` | 1e6 | RoPE 基础频率 |
| `tie_word_embeddings` | True | 嵌入绑定 |
| `rms_norm_eps` | 1e-6 | RMSNorm $\epsilon$ |
| `use_moe` | False | MoE 开关 |
| `num_experts` | 4 | 专家数(MoE) |
| `num_experts_per_tok` | 1 | top-1 路由 |
| `router_aux_loss_coef` | 5e-4 | 负载均衡系数 |

## 参数量计算公式

每层参数:

$$P_{\text{layer}} = \underbrace{d \cdot n_q \cdot d_h + 2 \cdot d \cdot n_{kv} \cdot d_h + n_q \cdot d_h \cdot d + 2 d_h}_{\text{Attention}} + \underbrace{3 \cdot d \cdot d_{\text{ff}}}_{\text{SwiGLU FFN}} + \underbrace{2d}_{\text{Norms}}$$

总参数(绑定权重):

$$P_{\text{total}} = V \cdot d + L \cdot P_{\text{layer}} + d$$

代入 $d=768, L=8, V=6400, n_q=8, n_{kv}=4, d_h=96, d_{\text{ff}}=2432$:

$$P_{\text{total}} = 4{,}915{,}200 + 8 \times 7{,}374{,}528 + 768 = 63{,}912{,}192 \approx 64\text{M}$$

In [ ]:
import math
d, L, V = 768, 8, 6400
n_q, n_kv, d_h = 8, 4, 96
d_ff = math.ceil(d * math.pi / 64) * 64  # 2432

P_attn = d*n_q*d_h + 2*d*n_kv*d_h + n_q*d_h*d + 2*d_h   # 1,769,664
P_ffn  = 3 * d * d_ff                                      # 5,603,328
P_norm = 2 * d                                             # 1,536
P_layer = P_attn + P_ffn + P_norm                          # 7,374,528
P_total = V * d + L * P_layer + d                          # 63,912,192

print(f"P_attn  = {P_attn:>12,}")
print(f"P_ffn   = {P_ffn:>12,}")
print(f"P_layer = {P_layer:>12,}")
print(f"P_total = {P_total:>12,} ≈ {P_total/1e6:.1f}M")

## Dense vs MoE 对比

| 指标 | Dense | MoE (4 experts) |
|---|---|---|
| 总参数 | 63.9M | 198.4M |
| 激活参数/token | 63.9M | 63.9M |
| FFN 参数/层 | 5.6M | 22.4M |
| 额外:router | — | 3,072 |

## Qwen3 对齐

minimind config 的核心字段与 `Qwen3Config` 逐字段匹配,可通过 `convert_model.py` 的 `load_state_dict(strict=True)` 直接转换,兼容 llama.cpp / vLLM / Ollama。